# AI Jobs Market Salaries initial cleaning
Tina Joshi | Group 2 | Data Engineering Milestone 2

This notebook reproduces the original-data inspection, initial cleaning, and quality evidence for **one structured dataset**. It retains all 1,500 original jobs, normalizes skill lists, and flags unresolved source conflicts. The PostgreSQL load is supplied separately as `sql/01_pgadmin_load_and_clean.sql` for pgAdmin.

Source: https://www.kaggle.com/datasets/alitaqishah/ai-jobs-market-2025-2026-salaries

**Source limitation:** the publisher says the dataset was compiled from salary reports and market research. Row-level original postings are not independently verified. Treat this as a compiled sample, and confirm its acceptability for the course's real-world-data requirement. This CSV does not supply the team's semi-structured/unstructured dataset.

AWS ingestion was subsequently verified from Tina’s pgAdmin screenshots: instance de-postgres, database ai_jobs_product, schema m2_ai_salaries. See evidence/aws_ingestion_verification.json.

In [1]:
from pathlib import Path
import hashlib, json, importlib.util, io, contextlib
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "raw").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "raw/ai_jobs_market_2025_2026.csv").is_file(), "Open this notebook from the unzipped package."
original_path = ROOT / "raw/ai_jobs_market_2025_2026.csv"
original_hash = hashlib.sha256(original_path.read_bytes()).hexdigest()
assert original_hash == "6ae7bc7bf724250cd8c977f86b620128318070a23cb0b17340e58d2321f7864e"
raw = pd.read_csv(original_path, dtype=str, keep_default_na=False)
print("Original dimensions:", raw.shape)
print("Original bytes:", original_path.stat().st_size)
print("SHA-256:", original_hash)
display(raw.head(3))

Original dimensions: (1500, 25)
Original bytes: 383900
SHA-256: 6ae7bc7bf724250cd8c977f86b620128318070a23cb0b17340e58d2321f7864e


,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,USA,On-site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,UK,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,USA,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k)


## Original quality checks
Read original values as strings so that parsing cannot silently replace source strings. Count blank cells, placeholder nulls, duplicate IDs, and full duplicate rows before processing.

In [2]:
missing = raw.apply(lambda x: x.str.strip().eq("")).sum()
print("Blank source cells:", int(missing.sum()))
print("Duplicate job IDs:", int(raw.job_id.duplicated().sum()))
print("Exact duplicate records:", int(raw.duplicated().sum()))
display(pd.DataFrame({"column": raw.columns, "blank_cells": missing.values, "distinct_values": raw.nunique().values}))

Blank source cells: 0
Duplicate job IDs: 0
Exact duplicate records: 0


,column,blank_cells,distinct_values
0,job_id,0,1500
1,job_title,0,25
2,job_category,0,12
3,experience_level,0,4
4,years_of_experience,0,15
5,education_required,0,5
6,annual_salary_usd,0,248
7,salary_min_usd,0,17
8,salary_max_usd,0,16
9,city,0,20


## Run the reproducible cleaning pipeline
The script validates numeric types and documented ranges, trims text, deduplicates skill tokens within each job, adds country codes and a derived monthly period, and flags salary and experience conflicts. Critical checks stop processing. Recoverable issues remain visible in the log; source values are retained in the raw layer.

Salary-tier boundaries: below $100k; $100k to below $150k; $150k to below $200k; $200k to $300k inclusive; above $300k. Original tier labels remain available for comparison. The derived posting date is the first day of the recorded month; the source does not contain an exact posting day.

In [3]:
spec = importlib.util.spec_from_file_location("salary_cleaning", ROOT / "scripts/clean_and_profile.py")
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
with contextlib.redirect_stdout(io.StringIO()):
    raw_again, cleaned, job_skills, checks, issues, summary = pipeline.run()
print(json.dumps(summary, indent=2))
display(checks.loc[checks.affected_count > 0, ["check_name", "affected_count", "severity", "decision"]])

{
  "dataset_version": 1,
  "raw_sha256": "6ae7bc7bf724250cd8c977f86b620128318070a23cb0b17340e58d2321f7864e",
  "raw_rows": 1500,
  "source_columns": 25,
  "cleaned_rows": 1500,
  "cleaned_columns": 40,
  "missing_cells": 0,
  "placeholder_null_cells": 0,
  "duplicate_rows": 0,
  "duplicate_ids": 0,
  "rows_with_repeated_skills": 119,
  "removed_skill_tokens": 120,
  "raw_skill_tokens": 9548,
  "job_skill_pairs": 9428,
  "unique_skill_keys": 93,
  "salary_range_conflicts": 288,
  "experience_conflicts": 1099,
  "tier_boundary_disagreements": 61,
  "global_rows": 82,
  "salary_analysis_rows": 1212,
  "country_salary_analysis_rows": 1155,
  "experience_consistent_rows": 401,
  "row_issue_log_entries": 1649,
  "first_period": "2025-01-01",
  "last_period": "2026-03-01",
  "role_count": 25,
  "country_label_count": 14,
  "actual_country_count": 13,
  "critical_checks_failed": 0,
  "raw_records_dropped": 0,
  "source_provenance_status": "Publisher says compiled from salary reports and job-m

,check_name,affected_count,severity,decision
43,repeated_skills_within_job,119,recoverable,"Remove repeated skill tokens within each job, ..."
44,annual_salary_outside_stated_range,288,recoverable,"Retain all source salary values, flag the conf..."
45,experience_label_years_conflict,1099,recoverable,Retain both fields and flag the conflict. Use ...
46,salary_tier_boundary_disagreement,61,recoverable,Retain source salary_tier and add salary_tier_...
47,global_geography_not_country,82,recoverable,"Retain Global listings, leave country_iso2 emp..."


## Show actual cleaning and unresolved conflicts
Repeated skills are removed within a job only. Different jobs that share a skill remain separate observations. Conflicting salaries and experience fields are flagged without inventing corrected values.

In [4]:
skill_examples = raw[["job_id", "required_skills"]].merge(
    cleaned[["job_id", "required_skills", "duplicate_skill_tokens_removed"]],
    on="job_id", suffixes=("_original", "_cleaned"))
display(skill_examples.loc[skill_examples.duplicate_skill_tokens_removed > 0].head(5))
display(cleaned.loc[~cleaned.salary_range_consistent,
    ["job_id", "annual_salary_usd", "salary_min_usd", "salary_max_usd", "annual_salary_usd_validated"]].head(5))
display(cleaned.loc[~cleaned.experience_consistent,
    ["job_id", "experience_level", "years_of_experience", "experience_consistent"]].head(5))
display(cleaned.loc[~cleaned.salary_tier_consistent,
    ["job_id", "annual_salary_usd", "salary_tier", "salary_tier_derived"]].head(5))

,job_id,required_skills_original,required_skills_cleaned,duplicate_skill_tokens_removed
3,AIJOB0004,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,1
10,AIJOB0011,Cloud|Python|Leadership|Enterprise Architectur...,Cloud|Python|Leadership|Enterprise Architecture,1
22,AIJOB0023,SQL|Communication|Python|Data Visualization|Bu...,SQL|Communication|Python|Data Visualization|Bu...,1
28,AIJOB0029,Communication|Python|SQL|Agile|Git|SQL,Communication|Python|SQL|Agile|Git,1
29,AIJOB0030,Research|Ethics Frameworks|Communication|Resea...,Research|Ethics Frameworks|Communication|Probl...,1


,job_id,annual_salary_usd,salary_min_usd,salary_max_usd,annual_salary_usd_validated
2,AIJOB0003,360000.0,160000,300000,NaN
4,AIJOB0005,283000.0,140000,260000,NaN
20,AIJOB0021,329000.0,160000,300000,NaN
21,AIJOB0022,272000.0,130000,260000,NaN
22,AIJOB0023,201000.0,95000,180000,NaN


,job_id,experience_level,years_of_experience,experience_consistent
1,AIJOB0002,Senior (6-9 yrs),2,False
2,AIJOB0003,Senior (6-9 yrs),4,False
3,AIJOB0004,Senior (6-9 yrs),3,False
4,AIJOB0005,Lead (10+ yrs),5,False
5,AIJOB0006,Entry (0-2 yrs),5,False


,job_id,annual_salary_usd,salary_tier,salary_tier_derived
59,AIJOB0060,100000.0,Entry (<$100k),Mid ($100-150k)
73,AIJOB0074,100000.0,Entry (<$100k),Mid ($100-150k)
117,AIJOB0118,150000.0,Mid ($100-150k),Upper-Mid ($150-200k)
123,AIJOB0124,150000.0,Mid ($100-150k),Upper-Mid ($150-200k)
204,AIJOB0205,100000.0,Entry (<$100k),Mid ($100-150k)


## Normalized skills and output reconciliation
One row in `job_skills.csv` represents one unique `(job_id, skill_key)` pair. `salary_analysis_ready.csv` is a conservative subset, while the main cleaned file retains all jobs. Global listings remain in the dataset but are excluded from physical-country comparisons.

In [5]:
assert len(raw) == len(cleaned) == 1500
assert not job_skills.duplicated(["job_id", "skill_key"]).any()
assert len(job_skills) == 9428
assert int(cleaned.duplicate_skill_tokens_removed.sum()) == 120
assert int((~cleaned.salary_range_consistent).sum()) == 288
assert int((~cleaned.experience_consistent).sum()) == 1099
assert int((~cleaned.salary_tier_consistent).sum()) == 61
assert hashlib.sha256(original_path.read_bytes()).hexdigest() == original_hash
print("All reconciliation checks passed. Original CSV bytes are unchanged.")
print("Cleaned rows:", len(cleaned), "Job-skill pairs:", len(job_skills))
print("Conservative salary rows:", int(cleaned.salary_analysis_ready.sum()))
print("Physical-country salary rows:", int(cleaned.country_salary_analysis_ready.sum()))
display(job_skills.head(10))
display(issues.head(10))

All reconciliation checks passed. Original CSV bytes are unchanged.
Cleaned rows: 1500 Job-skill pairs: 9428
Conservative salary rows: 1212
Physical-country salary rows: 1155


,job_id,skill_key,skill
0,AIJOB0001,apis,APIs
1,AIJOB0001,planning systems,Planning Systems
2,AIJOB0001,python,Python
3,AIJOB0001,cloud,Cloud
4,AIJOB0001,sql,SQL
5,AIJOB0001,leadership,Leadership
6,AIJOB0002,python,Python
7,AIJOB0002,documentation,Documentation
8,AIJOB0002,llm apis,LLM APIs
9,AIJOB0002,prompt design,Prompt Design


,job_id,source_row,issue,severity,original_value,processed_value,action
0,AIJOB0002,2,experience_label_years_conflict,recoverable,Senior (6-9 yrs);years=2,experience_consistent=false,Kept both source values; excluded from agreeme...
1,AIJOB0003,3,annual_salary_outside_stated_range,recoverable,annual=360000.0;min=160000;max=300000,annual_salary_usd_validated=NULL,Kept original values; flagged; excluded from c...
2,AIJOB0003,3,experience_label_years_conflict,recoverable,Senior (6-9 yrs);years=4,experience_consistent=false,Kept both source values; excluded from agreeme...
3,AIJOB0004,4,repeated_skills_within_job,recoverable,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,Deduplicated repeated tokens within this job o...
4,AIJOB0004,4,experience_label_years_conflict,recoverable,Senior (6-9 yrs);years=3,experience_consistent=false,Kept both source values; excluded from agreeme...
5,AIJOB0005,5,annual_salary_outside_stated_range,recoverable,annual=283000.0;min=140000;max=260000,annual_salary_usd_validated=NULL,Kept original values; flagged; excluded from c...
6,AIJOB0005,5,experience_label_years_conflict,recoverable,Lead (10+ yrs);years=5,experience_consistent=false,Kept both source values; excluded from agreeme...
7,AIJOB0006,6,experience_label_years_conflict,recoverable,Entry (0-2 yrs);years=5,experience_consistent=false,Kept both source values; excluded from agreeme...
8,AIJOB0008,8,experience_label_years_conflict,recoverable,Mid (3-5 yrs);years=7,experience_consistent=false,Kept both source values; excluded from agreeme...
9,AIJOB0008,8,global_geography_not_country,recoverable,Global,country_iso2=NULL,Retained listing; excluded from physical-count...


## Verified AWS ingestion
The SQL has now been run successfully in Amazon RDS instance **de-postgres**, database **ai_jobs_product**, schema **m2_ai_salaries**. User-provided pgAdmin screenshots confirm 1,500 raw records, 1,500 cleaned records, 9,428 job-skill pairs, the eight expected quality checks, and the AIJOB0004 skill-cleaning example.

Use `sql/02_demo_and_verification_queries.sql` for read-only inspection and the video. The load script is for a fresh target schema; it should not be rerun in the populated schema. No direct connection to the user database was made by the notebook. Local PostgreSQL validation is separate.

The full team submission still needs the other dataset sections, updated proposal, meeting-notes link, and an accessible narrated five-minute video link. Access by teammates to this AWS database has not been verified.